# Computing the module of points

In [1]:
load('functions.sage')

## Setting

Let's create the base finite fields, and the polynomial ring acting as the *function ring*:

In [2]:
Fq = GF(3)
A.<T> = Fq[]
Fpp.<zpp> = Fq.extension(2)
K.<z> = Fpp.extension(8)

And we now create a Drinfeld module $\phi$:

In [3]:
phi = DrinfeldModule(A, [K(zpp), 1, K(zpp)])
phi

Drinfeld module defined by T |--> (z^15 + z^14 + 2*z^13 + 2*z^11 + 2*z^10 + 2*z^8 + 2*z^7 + 2*z^5 + 2*z^4 + 2*z^3 + 2)*τ^2 + τ + z^15 + z^14 + 2*z^13 + 2*z^11 + 2*z^10 + 2*z^8 + 2*z^7 + 2*z^5 + 2*z^4 + 2*z^3 + 2

Here, `K` is a degree $16$ extension of the field with $3$ elements. It is defined by a generator `z` over $\mathbb F_q$. We fix an $\mathbb F_q$-basis of $K$ once and for all:

In [4]:
basis_K = phi.base_over_constants_field().basis_over(Fq)

**Remark.** *We mention that the Ore polynomial ring relative to $\mathbb F_q$ and $K$ is automatically created by the `DrinfeldModule` constructor. So is the morphism $\gamma: \mathbb F_q[T]\to K$, which is automatically defined as the map that sends $T$ to the constant coefficient of $\phi_T$, here, the image in $K$ of `zpp`.*

## Computing the invariant factors and a Frobenius decomposition of $\phi(K)$

First, we showcase compute the module of rational points of the Drinfeld module:

In [5]:
invariant_factors, frobenius_decomposition = module_of_points_invariants(phi, basis_K, frobenius=True)

<string>:132: FutureWarning: This class/method/function is marked as experimental. It, its functionality or its interface might change without a formal deprecation.
See https://github.com/sagemath/sage/issues/13215 for details.


(Please do not pay attention to the warning.)

<a id='cell_remark'></a> **Remark.** *The user will notice that the computation is slow. This has to do with the SageMath native implementation for computing the change-of-basis matrix between the matrix of $\phi_T$ and its Frobenius form. Computing the Frobenius form itself is almost instant, but the change-of-basis matrix unfortunately has to be computed separately and after, adding significant overhead. We wish to solve that issue in the future, for example by implementing Storjohann's algorithm. It is also possible not to ask for the Frobenius decomposition, for a much faster computation.*

We observe that the invariant factors are $T^6 + 2T^5 + T^4 + T^3 + 2T$ and $T^{10} + 2T^9 + 2T^2 + T$:

In [6]:
[d1, d2] = invariant_factors
print(d1)
print(d2)

T^6 + 2*T^5 + T^4 + T^3 + 2*T
T^10 + 2*T^9 + 2*T^2 + T


In [7]:
[x1, x2] = frobenius_decomposition
print(x1)
print(x2)

z^15 + 2*z^14 + 2*z^12 + 2*z^11 + 2*z^10 + z^9 + 2*z^7 + z^6 + 2*z^4 + z^3 + z^2
z^15 + 2*z^11 + z^10 + 2*z^9 + z^8 + z^7 + z^4 + 2


In [8]:
invariant_factors = module_of_points_invariants(phi, basis_K, frobenius=False)

We verify that the invariant factors form a divisibility chain:

In [9]:
assert d1.divides(d2)

And that their product is the $\mathbb F_q[T]$-*number of points* (*i.e.* the Euler-Poincaré characteristic of $\phi(K)$):

In [10]:
number_of_points = phi.frobenius_charpoly()(1)
assert number_of_points == prod(invariant_factors)

We verify that the elements of the Frobenius decomposition have annihilators `d1` and `d2`, respectively:

In [11]:
assert order(phi, x1) == d1
assert order(phi, x2) == d2

We verify that the two elements of the Frobenius decomposition are $\mathbb F_q[T]$-linearily independent (warning: long computation):

In [12]:
for a in A.polynomials(d1.degree()):
    if phi(a)(x1) == d2:
        raise ValueError("The Frobenius decomposition is incorrect")

## Computing the invariant factors and the Frobenius decomposition of the kernel of an isogeny

Then we compute the kernel of an isogeny. First we define an isogeny:

In [13]:
t = phi.ore_variable()
isogeny_pol = phi(T^3 + T + 1).right_gcd(t^2 - phi(T))
isogeny = phi.hom(isogeny_pol)
print(isogeny)

Drinfeld Module morphism:
  From: Drinfeld module defined by T |--> (z^15 + z^14 + 2*z^13 + 2*z^11 + 2*z^10 + 2*z^8 + 2*z^7 + 2*z^5 + 2*z^4 + 2*z^3 + 2)*τ^2 + τ + z^15 + z^14 + 2*z^13 + 2*z^11 + 2*z^10 + 2*z^8 + 2*z^7 + 2*z^5 + 2*z^4 + 2*z^3 + 2
  To:   Drinfeld module defined by T |--> (2*z^15 + 2*z^14 + z^13 + z^11 + z^10 + z^8 + z^7 + z^5 + z^4 + z^3 + 2)*τ^2 + z^15 + z^14 + 2*z^13 + 2*z^11 + 2*z^10 + 2*z^8 + 2*z^7 + 2*z^5 + 2*z^4 + 2*z^3 + 2
  Defn: τ + 2*z^15 + 2*z^14 + z^13 + z^11 + z^10 + z^8 + z^7 + z^5 + z^4 + z^3


In [14]:
invariant_factors, frobenius_decomposition = morphism_kernel_invariants(isogeny, basis_K, frobenius=True)
print(invariant_factors)
print(frobenius_decomposition)

[T + 2]
[2*z^15 + 2*z^14 + z^13 + z^11 + z^10 + z^8 + z^7 + z^5 + z^4 + z^3 + 1]


**Remark.** To explain the running time of the the above *cell*, see [here](#cell_remark).

This time, we observe that the kernel of the isogeny is a cyclic module isomorphic to $\mathbb F_q[T]/(T + 2)$. This is not surprising: the isogeny has $\tau$-degree one, so its kernel is either trivial or one-dimensional over $\mathbb F_q$.

In [15]:
[d1] = invariant_factors
[x1] = frobenius_decomposition

We verify that the order of the element `x1` is its corresponding invariant factor:

In [16]:
assert order(phi, x1) == d1

And in that case, the unique invariant factor is the *norm* of the isogeny:

In [17]:
assert isogeny_pol.right_divides(phi(d1))
assert isogeny.norm().gen() == d1

## The special case of the torsion

We now look at the special case of the torsion. First of all, we can compute the maximal separable rational torsion:

In [18]:
g_phi = maximal_rational_separable_torsion(phi)
print(g_phi)
print(g_phi.factor())

T^6 + 2*T^5 + T^4 + T^3 + 2*T
T * (T + 1) * (T^2 + 1) * (T^2 + T + 2)


All the divisors $a$ of $g_\phi$ have a rational torsion:

In [19]:
invariant_factors_g_phi = morphism_kernel_invariants(phi.hom(g_phi), basis_K)
print(invariant_factors_g_phi)

[T^6 + 2*T^5 + T^4 + T^3 + 2*T, T^6 + 2*T^5 + T^4 + T^3 + 2*T]


We verify that the $g_\phi$ torsion is isomorphic to $(\mathbb F_q[T]/(g_\phi))^2$, which is always true over the algebraic closure for a Drinfeld module of rank two and for $\phi_{g_\phi}$ separable.

In [20]:
assert invariant_factors_g_phi == [g_phi, g_phi]

If we take a polynomial $a \in \mathbb F_q[T]$ that is not a divisor of $g_\phi$, then the torsion is not fully rationnal. For $a = g_\phi + 1$, $\phi_a$ is still separable but has only one root in $K$:

In [21]:
print(morphism_kernel_invariants(phi.hom(g_phi+1), basis_K))

[]


For $a = T * g_\phi$, we observe that $\phi(K)[a]$ is not isomorphic to $(\mathbb F_q[T]/(a))^2$:

In [22]:
print(morphism_kernel_invariants(phi.hom(g_phi*T), basis_K))

[T^6 + 2*T^5 + T^4 + T^3 + 2*T, T^6 + 2*T^5 + T^4 + T^3 + 2*T]


In [23]:
assert morphism_kernel_invariants(phi.hom(g_phi*T), basis_K) != [g_phi*T, g_phi*T]

We can also study the $n_\phi$-torsion, where $n_\phi$ is the *$\mathbb F_q[T]$-number of points of $\phi(K)$*, formally defined as the *Euler-Poincaré characteristic* of $\phi(K)$:

In [24]:
n_phi = phi.frobenius_charpoly()(1)
print(n_phi)

T^16 + T^15 + 2*T^14 + 2*T^12 + 2*T^11 + T^10 + 2*T^8 + 2*T^7 + T^6 + T^4 + T^3 + 2*T^2


It is always true that $g_\phi$ divides $n_\phi$. However, those two quantities are never equal for Drinfeld modules of rank at least two.

In [25]:
assert g_phi.divides(n_phi)
assert g_phi != n_phi

And indeed one can verify that in our case $\phi(K)[n_\phi]$ and $\phi(K)$ are isomorphic, and that $\phi[n_\phi]$ is strictly larger than $\phi(K)[n_\phi]$:

In [26]:
invariant_factors_points = module_of_points_invariants(phi, basis_K)
assert invariant_factors_points == morphism_kernel_invariants(phi.hom(n_phi), basis_K) != [n_phi, n_phi]